### Import Dependencies

In [15]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

### Download all data from Qdrant

In [16]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [17]:
all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False   
)

In [20]:
all_points[0][0].payload

{'preprocessed_description': "Now That's What I Call A Love Song / Various ",
 'image': 'https://m.media-amazon.com/images/I/51QqhSYl+DL.jpg',
 'rating_number': 120,
 'price': 25.68,
 'average_rating': 4.5,
 'parent_asin': 'B0BS1SHT91'}

In [21]:
all_points[0]

[Record(id=0, payload={'preprocessed_description': "Now That's What I Call A Love Song / Various ", 'image': 'https://m.media-amazon.com/images/I/51QqhSYl+DL.jpg', 'rating_number': 120, 'price': 25.68, 'average_rating': 4.5, 'parent_asin': 'B0BS1SHT91'}, vector=None, shard_key=None, order_value=None),
 Record(id=1, payload={'preprocessed_description': 'If ', 'image': 'https://m.media-amazon.com/images/I/51433HXqQrL.jpg', 'rating_number': 167, 'price': 40.74, 'average_rating': 4.6, 'parent_asin': 'B0C61QHRB6'}, vector=None, shard_key=None, order_value=None),
 Record(id=2, payload={'preprocessed_description': 'Live in New York 1979--The Ultimate Edition ', 'image': 'https://m.media-amazon.com/images/I/41zdV6jHPEL.jpg', 'rating_number': 138, 'price': 24.99, 'average_rating': 4.9, 'parent_asin': 'B09YPXFPQY'}, vector=None, shard_key=None, order_value=None),
 Record(id=3, payload={'preprocessed_description': 'Amazing Grace ', 'image': 'https://m.media-amazon.com/images/I/315i1aDSDlL.jpg', '

In [23]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in all_points[0]]

In [24]:
all_context

[{'id': 'B0BS1SHT91', 'text': "Now That's What I Call A Love Song / Various "},
 {'id': 'B0C61QHRB6', 'text': 'If '},
 {'id': 'B09YPXFPQY', 'text': 'Live in New York 1979--The Ultimate Edition '},
 {'id': 'B09XFF59GL', 'text': 'Amazing Grace '},
 {'id': 'B0BJ7P5GP5',
  'text': 'Things Happen That Way - Exclusive Limited Edition Blue Colored Vinyl LP '},
 {'id': 'B0BMXSPF6M', 'text': 'Santa Baby '},
 {'id': 'B0B2C1N4XL', 'text': 'Nowhere Generation II '},
 {'id': 'B09X28P6QN', 'text': 'Coming Home '},
 {'id': 'B0BYB5VV46', 'text': 'Coming Home '},
 {'id': 'B0B61K8PQM',
  'text': 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman [4 LP] '},
 {'id': 'B0BM4XNTW7', 'text': 'On The Prowl       Explicit Lyrics '},
 {'id': 'B0BG2ZDW9P', 'text': 'Love '},
 {'id': 'B09X4NNL7Q',
  'text': 'Die Sehnsucht Ist Mein Steuermann: Das Beste Aus 10 Jahren - Deluxe '},
 {'id': 'B09X4NNL7Q',
  'text': 'Die Sehnsucht Ist Mein Steuermann: Das Beste Aus 10 Jahren - Deluxe '},
 {'id': 'B0B6GJYWXQ',

### Render a prompt to generate synthetic Eval reference dataset

In [36]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}


SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

In [27]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, 

In [28]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[
  {
    "id": "B0BS1SHT91",
    "text": "Now That's What I Call A Love Song / Various "
  },
  {
    "id": "B0C61QHRB6",
    "text": "If "
  },
  {
    "id": "B09YPXFPQY",
    "text": "Live in New York 1979--The Ultimate Edition "
  },
  {
    "id": "B09XFF59GL",
    "text": "Amazing Grace "
  },
  {
    "id": "B0BJ7P5GP5",
    "text": "Things Happen That Way - Exclusive Limited Edition Blue Colored Vinyl LP "
  },
  {
    "id": "B0BMXSPF6M",
    "text": "Santa Baby "
  },
  {
    "id": "B0B2C1N4XL",
    "text": "Nowhere Generation II "
  },
  {
    "id": "B09X28P6QN",
    "text": "Coming Home "
  },
  {
    "id": "B0BYB5VV46",
    "text": "Coming Home "
  },
  {
    "id": "B0B61K8PQM",
    "text": "All My Friends: Celebrating The Songs & Voice Of Gregg Allman [4 LP] "
  },
  {
    "id": "B0BM4XNTW7",
    "text": "On The Prowl       Explicit Lyrics "
  },
  {
    "id": "B0BG2ZDW9P",
    "text": "Love "
 

In [29]:
response = openai.chat.completions.create(
    model="gpt-5.4",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

[
  {
    "reasoning": "This can be answered from a single product title because the title explicitly contains both the format and the artist tribute description.",
    "question": "Do you have 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman' on vinyl, and if so how many LPs is it?",
    "chunk_ids": ["B0B61K8PQM"],
    "answer_example": "Yes, we have 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman' and it is listed as a 4 LP edition.",
    "answer_example": "Yes, we have 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman' and it is listed as a 4 LP edition."
  },
  {
    "reasoning": "The product title directly states the edition and colored vinyl format, so one product is enough.",
    "question": "Is 'Things Happen That Way' available as a limited edition blue colored vinyl LP?",
    "chunk_ids": ["B0BJ7P5GP5"],
    "answer_example": "Yes, we have 'Things Happen That Way - Exclusive Limited Edition Blue Colored Vinyl LP' available."
  },
  {

In [30]:
response.usage

CompletionUsage(completion_tokens=3313, prompt_tokens=1923, total_tokens=5236, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

In [31]:
json_output = response.choices[0].message.content

In [32]:
json_output = json_output.replace("```json", "").replace("```", "")

In [33]:
print(json_output)

[
  {
    "reasoning": "This can be answered from a single product title because the title explicitly contains both the format and the artist tribute description.",
    "question": "Do you have 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman' on vinyl, and if so how many LPs is it?",
    "chunk_ids": ["B0B61K8PQM"],
    "answer_example": "Yes, we have 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman' and it is listed as a 4 LP edition.",
    "answer_example": "Yes, we have 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman' and it is listed as a 4 LP edition."
  },
  {
    "reasoning": "The product title directly states the edition and colored vinyl format, so one product is enough.",
    "question": "Is 'Things Happen That Way' available as a limited edition blue colored vinyl LP?",
    "chunk_ids": ["B0BJ7P5GP5"],
    "answer_example": "Yes, we have 'Things Happen That Way - Exclusive Limited Edition Blue Colored Vinyl LP' available."
  },
  {

In [34]:
json_output = json.loads(json_output)

In [35]:
json_output

[{'reasoning': 'This can be answered from a single product title because the title explicitly contains both the format and the artist tribute description.',
  'question': "Do you have 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman' on vinyl, and if so how many LPs is it?",
  'chunk_ids': ['B0B61K8PQM'],
  'answer_example': "Yes, we have 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman' and it is listed as a 4 LP edition."},
 {'reasoning': 'The product title directly states the edition and colored vinyl format, so one product is enough.',
  'question': "Is 'Things Happen That Way' available as a limited edition blue colored vinyl LP?",
  'chunk_ids': ['B0BJ7P5GP5'],
  'answer_example': "Yes, we have 'Things Happen That Way - Exclusive Limited Edition Blue Colored Vinyl LP' available."},
 {'reasoning': "The title itself contains the format detail '[2 CD]', so this is answerable from one product.",
  'question': "Do you stock 'Licked Live In NYC' on 2 CD?",
 

In [37]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

In [38]:
print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

Single chunk questions: 15
Multiple chunk questions: 10
Impossible questions: 5


In [42]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B0BS1SHT91")
            )
        ]
    )
)[0]

In [43]:
points[0].payload

{'preprocessed_description': "Now That's What I Call A Love Song / Various ",
 'image': 'https://m.media-amazon.com/images/I/51QqhSYl+DL.jpg',
 'rating_number': 120,
 'price': 25.68,
 'average_rating': 4.5,
 'parent_asin': 'B0BS1SHT91'}

In [44]:
def get_description(parent_asin: str) -> str:
    
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]

    return points[0].payload["preprocessed_description"]

In [45]:
get_description("B0BS1SHT91")

"Now That's What I Call A Love Song / Various "

### Create Eval dataset in LangSmith

In [46]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [47]:
ls_client = Client()

In [48]:
dataset_name = "rag-evaluation-dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset"
)

In [49]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"]
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )